In [1]:
import pandas as pd

In [2]:
Stocks = pd.read_excel("onglets\Data_set.xlsx", index_col=0, parse_dates=True)
Stocks_id = Stocks.columns
len(Stocks_id)

15

In [26]:
Stocks = pd.read_excel("onglets\Data_set.xlsx", parse_dates=True)
Stocks = Stocks.sort_values("Date").reset_index(drop=True)
Stocks.set_index("Date", inplace=True)

Stocks_id = Stocks.columns
Stocks_id

Index(['LVMH', 'OREP', 'PRTP', 'AirbusSE', 'Safran SA', 'Air Liquide SA',
       'BOLL_Price', 'ENGIE_Price', 'TTEF_Price', 'DSY (Dassault)',
       'STMPA (STMicroelectornics)', 'Capegimini', 'BNPP', 'CAGR', 'AXA SA'],
      dtype='object')

In [27]:
Stocks = Stocks.asfreq('B')
#Stocks = Stocks.fillna(method='ffill') 

In [28]:
Stocks[['LVMH', 'OREP']].head(30)

,LVMH,OREP
Date,,
2021-01-22,511.5,296.3
2021-01-25,501.9,299.0
2021-01-26,508.0,301.8
2021-01-27,506.4,297.6
2021-01-28,517.0,297.4
2021-01-29,498.3,290.1
2021-02-01,507.7,296.5
2021-02-02,524.9,298.4
2021-02-03,524.5,295.9


In [23]:
Stocks.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1326 entries, 2021-01-22 to 2026-02-20
Freq: B
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   LVMH                        1326 non-null   float64
 1   OREP                        1326 non-null   float64
 2   PRTP                        1326 non-null   float64
 3   AirbusSE                    1326 non-null   float64
 4   Safran SA                   1326 non-null   float64
 5   Air Liquide SA              1326 non-null   float64
 6   BOLL_Price                  1326 non-null   float64
 7   ENGIE_Price                 1326 non-null   float64
 8   TTEF_Price                  1326 non-null   float64
 9   DSY (Dassault)              1326 non-null   float64
 10  STMPA (STMicroelectornics)  1326 non-null   float64
 11  Capegimini                  1326 non-null   float64
 12  BNPP                        1326 non-null   float64
 13  CAGR   

In [24]:
Stocks.isna().sum()

LVMH                          0
OREP                          0
PRTP                          0
AirbusSE                      0
Safran SA                     0
Air Liquide SA                0
BOLL_Price                    0
ENGIE_Price                   0
TTEF_Price                    0
DSY (Dassault)                0
STMPA (STMicroelectornics)    0
Capegimini                    0
BNPP                          0
CAGR                          0
AXA SA                        0
dtype: int64

In [ ]:
# Définir fréquence business day (optionnel)
#data = data.asfreq('B')
#data = data.fillna(method='ffill')  # remplir les jours manquants

# liste des actions (toutes les colonnes sauf Date)

# stockage résultats VaR
VaR_results = []
Ljung_Box_rendements = {}
Resultat_GARCH11 = {}
Resultat_GJR_GARCH = {}
# création PDF
with PdfPages("Diagnostics_Portofolio_Assets.pdf") as pdf:

    # -------------------------------------------------
    # BOUCLE SUR CHAQUE ACTION
    # -------------------------------------------------

    for col in price_columns:
        # -------------------------------------------------
        # 2. Calcul rendement
        # -------------------------------------------------

        #returns = returns.dropna()
        # Afficher les 5 premières lignes
        print(f"\nPremières lignes des rendements pour {col}:")
        print(returns.head())
        returns_scaled = returns * 100
        print(f"\nStatistiques descriptives {col}")
        print(returns.describe())

        # -------------------------------------------------
        # 3. ACF et PACF
        # -------------------------------------------------

        fig, ax = plt.subplots()
        plot_acf(returns, lags=40, ax=ax)
        ax.set_title(f"ACF_{col}_Returns")
        pdf.savefig(fig)
        plt.close(fig)

        fig, ax = plt.subplots()
        plot_pacf(returns, lags=40, ax=ax)
        ax.set_title(f"PACF_{col}_Returns")
        pdf.savefig(fig)
        plt.close(fig)

        # -------------------------------------------------
        # 4. Ljung Box rendements
        # -------------------------------------------------

        lb_test = acorr_ljungbox(returns, lags=[10], return_df=True)

        print(f"\nTest Ljung Box Rendements {col}")
        print(lb_test)

        alpha = 0.05
        alpha_var = 1 - confidence
        Ljung_Box_rendements = lb_test
        # -------------------------------------------------
        # CAS 1 : AUTOCORRELATION
        # -------------------------------------------------

        if lb_test['lb_pvalue'].iloc[0] < alpha:

            print("Autocorrélation détectée → AR(1)")

            model_ar = ARIMA(returns, order=(1,0,0)).fit()

            print(model_ar.summary())

            residuals = model_ar.resid

    # -------------------------------------------------
    # Ljung-Box sur résidus
    # -------------------------------------------------

            lb_res = acorr_ljungbox(residuals, lags=[10], return_df=True)

            print("\nTest Ljung-Box sur résidus AR(1)")
            print(lb_res)

            if lb_res['lb_pvalue'].iloc[0] < alpha:

                print("Autocorrélation n'est pas supprimé avec AR(1) -> Augmentation p")

                # initialisation
                p = 1  # ordre AR initial
                alpha = 0.05
                max_p = 5
                while p <= max_p:
                        model = ARIMA(returns, order=(p,0,0)).fit()
                        lb_res = acorr_ljungbox(model.resid, lags=[10], return_df=True)
                        if lb_res['lb_pvalue'].iloc[0] > 0.05:
                            print(f"Autocorrélation supprimée avec AR({p})")
                            break
                        p += 1
                else:
                        print("Résidus toujours autocorrélés → utiliser ARMA(1,1)")
                        model = ARIMA(returns, order=(1,0,1)).fit()
                        if lb_res['lb_pvalue'].iloc[0] > 0.05:
                            print(f"Autocorrélation supprimée avec ARMA({1,1})")
                        else:
                            print("La moyenne est mal spécifiée après AR(5) et ARMA(1,1)")
                            sys.exit()
            else:

                print("La moyenne est correctement modélisée")
        
            # -------------------------------------------------
            # Test sur résidus carrés
            # -------------------------------------------------

            lb_sq = acorr_ljungbox(residuals**2, lags=[10], return_df=True)

            if lb_sq['lb_pvalue'].iloc[0] < alpha:

                    print("Effet ARCH détecté")

                    # -------------------
                    # GARCH(1,1) Normal
                    # -------------------

                    garch = arch_model(returns_scaled, mean='AR', lags=1,
                                    vol='GARCH', p=1, q=1,
                                    dist='normal')

                    garch_fit = garch.fit(disp="off")
                    
                    print(f"\nRésultat GARCH(1,1)_{col}:", garch_fit.summary()) 
                    Resultat_GARCH11[col] = garch_fit.summary()
                    # -------------------
                    # GJR GARCH Student
                    # -------------------

                    gjr = arch_model(returns_scaled, mean='AR', lags=1,
                                    vol='GARCH', p=1, o=1, q=1,
                                    dist='t')

                    gjr_fit = gjr.fit(disp="off")
                    
                    print(f"\nRésultat GJR-GARCH_{col}:", gjr_fit.summary()) 
                    Resultat_GJR_GARCH = gjr_fit.summary()
                    # -------------------
                    # VaR
                    # -------------------

                    sigma_garch = np.sqrt(garch_fit.forecast(horizon=1).variance.iloc[-1,0])/100
                    #mu_garch = garch_fit.params['mu']
                    mu_garch = garch_fit.params.get('mu', garch_fit.params.get('Const', 0))/100

                    var_garch = mu_garch + sigma_garch * norm.ppf(alpha_var)

                    sigma_gjr = np.sqrt(gjr_fit.forecast(horizon=1).variance.iloc[-1,0])/100
                    #mu_gjr = gjr_fit.params['mu']
                    mu_gjr= gjr_fit.params.get('mu', gjr_fit.params.get('Const', 0))/100
                    df = gjr_fit.params['nu']

                    var_gjr = mu_gjr + sigma_gjr * t.ppf(alpha_var, df)
                    
                    print(f"VaR_GARCH_{col} :", var_garch)
                    print(f"VaR_GJR_{col} :", var_gjr)

                    VaR_results.append([col,"GARCH",var_garch])
                    VaR_results.append([col,"GJR",var_gjr])

            else:

                    print("Variance constante → VaR paramétrique")

                    mu = returns.mean()
                    sigma = returns.std()

                    var_param = mu + sigma * norm.ppf(alpha_var)

                    print(f"VaR_Parametric_{col} :", var_param)

                    VaR_results.append([col,"Parametric",var_param])

        # -------------------------------------------------
        # CAS 2 : PAS AUTOCORRELATION
        # -------------------------------------------------

        else:

            print("Pas autocorrélation rendements")

            lb_sq = acorr_ljungbox(returns**2, lags=[10], return_df=True)

            if lb_sq['lb_pvalue'].iloc[0] < alpha:

                print("Effet ARCH détecté → GARCH")

                garch = arch_model(returns_scaled, mean='Constant',
                                vol='GARCH', p=1, q=1,
                                dist='normal')

                garch_fit = garch.fit(disp="off")

                print(f"\nRésultat GARCH(1,1)_{col}:", garch_fit.summary())

                gjr = arch_model(returns_scaled, mean='Constant',
                                vol='GARCH', p=1, o=1, q=1,
                                dist='t')

                gjr_fit = gjr.fit(disp="off")

                print(f"\nRésultat GJR-GARCH_{col}:", gjr_fit.summary())

                # -------------------------------------------------
                # Calcul VaR
                # -------------------------------------------------

                sigma_garch = np.sqrt(garch_fit.forecast(horizon=1).variance.iloc[-1,0])/100
                mu = returns.mean()/100

                var_garch = mu + sigma_garch * norm.ppf(alpha_var)

                sigma_gjr = np.sqrt(gjr_fit.forecast(horizon=1).variance.iloc[-1,0])/100
                df = gjr_fit.params['nu']

                var_gjr = mu + sigma_gjr * t.ppf(alpha_var, df)

                print(f"VaR_GARCH_{col} :", var_garch)
                print(f"VaR_GJR_{col} :", var_gjr)

                VaR_results.append([col,"GARCH",var_garch])
                VaR_results.append([col,"GJR",var_gjr])

            else:

                print("Variance constante → VaR paramétrique simple")

                mu = returns.mean()
                sigma = returns.std()

                var_param = mu + sigma * norm.ppf(alpha_var)

                print(f"VaR_Parametric_{col} :", var_param)

                VaR_results.append([col,"Parametric",var_param])

# -------------------------------------------------
# 5. Tableau final VaR
# -------------------------------------------------

VaR_table = pd.DataFrame(VaR_results,
                        columns=["Asset","Model","VaR"])

print("\nTableau final VaR")
print(VaR_table)

return VaR_table, Ljung_Box_rendements, Resultat_GARCH11, Resultat_GJR-GARCH

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from matplotlib.backends.backend_pdf import PdfPages

import sys

from arch import arch_model
from scipy.stats import norm, t

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


def plot_acf_pacf(series, col_name, pdf, lags=40, suffix="Returns"): #series = returns, col_name = portfolio
    """
    Trace les diagrammes ACF et PACF pour une série et les enregistre dans un PDF.
    
    Parameters:
    -----------
    series : pd.Series ou np.array
        La série de données à analyser (ex: rendements ou rendements carrés)
    col_name : str
        Nom de la colonne / actif
    pdf : PdfPages
        Objet PdfPages pour sauvegarder les figures
    lags : int, optionnel
        Nombre de lags à afficher (défaut=40)
    suffix : str, optionnel
        Texte à ajouter dans le titre et le nom des figures (ex: "Returns", "Residuals²")
    """
    
    # Diagramme ACF
    fig, ax = plt.subplots(figsize=(8,4))
    plot_acf(series, lags=lags, ax=ax)
    ax.set_title(f"ACF_{col_name}_{suffix}")
    pdf.savefig(fig)
    plt.close(fig)
    
    # Diagramme PACF
    fig, ax = plt.subplots(figsize=(8,4))
    plot_pacf(series, lags=lags, ax=ax, method='ywm')  # méthode Yule-Walker modifiée
    ax.set_title(f"PACF_{col_name}_{suffix}")
    pdf.savefig(fig)
    plt.close(fig)

def var_parametric(returns, confidence):
    """
    Calcule la VaR paramétrique simple pour rendements avec moyenne et variance constantes.
    """
    mu = returns.mean()
    sigma = returns.std()
    var_param = mu + sigma * norm.ppf(confidence)
    return var_param


une fonction pour faire le plot du ACF et du PACF

In [ ]:
#----------------------------------------------
# Back-testing
#----------------------------------------------

def compute_var_final_table(VaR_table):

    results = []

    for asset in VaR_table['Asset'].unique():

        asset_rows = VaR_table[VaR_table['Asset'] == asset]

        has_garch = asset_rows['Variance_Model'].str.contains("GARCH").any()

        candidates = []

        for _, r in asset_rows.iterrows():

            if not pd.isna(r['VaR']):
                candidates.append((r['VaR'], r['Variance_Model']))

            if not pd.isna(r['EVT_VaR']):
                candidates.append((r['EVT_VaR'], "EVT-" + r['Variance_Model']))

        if len(candidates) > 0:

            var_final, model_name = min(candidates, key=lambda x: x[0])

        else:

            var_final = None
            model_name = None

        results.append([asset, var_final, model_name])

    VaR_final_table = pd.DataFrame(
        results,
        columns=["Asset", "VaR_final", "VaR_model"]
    )

    return VaR_final_table


VaR_final_table = compute_var_final_table(VaR_table)

print("\nVaR finale par action\n")
print(VaR_final_table)



import pandas as pd
import numpy as np
from scipy.stats import chi2
import pandas as pd
import numpy as np
from scipy.stats import chi2


def kupiec_test(n, x, alpha):

    p_hat = x / n

    L0 = (1 - alpha)**(n - x) * alpha**x
    L1 = (1 - p_hat)**(n - x) * p_hat**x

    LR = -2 * np.log(L0 / L1)

    p_value = 1 - chi2.cdf(LR, df=1)

    return p_value


def christoffersen_test(returns, var_series):

    violations = (returns < var_series).astype(int)

    n00 = n01 = n10 = n11 = 0

    for t in range(1, len(violations)):

        if violations.iloc[t-1] == 0 and violations.iloc[t] == 0:
            n00 += 1
        elif violations.iloc[t-1] == 0 and violations.iloc[t] == 1:
            n01 += 1
        elif violations.iloc[t-1] == 1 and violations.iloc[t] == 0:
            n10 += 1
        else:
            n11 += 1

    n0 = n00 + n01
    n1 = n10 + n11

    pi0 = n01 / n0 if n0 > 0 else 0
    pi1 = n11 / n1 if n1 > 0 else 0
    pi = (n01 + n11) / (n0 + n1) if (n0 + n1) > 0 else 0

    L0 = ((1 - pi)**(n00 + n10)) * (pi**(n01 + n11))
    L1 = ((1 - pi0)**n00) * (pi0**n01) * ((1 - pi1)**n10) * (pi1**n11)

    if L0 > 0 and L1 > 0:
        LR = -2 * np.log(L0 / L1)
    else:
        LR = 0

    p_value = 1 - chi2.cdf(LR, df=1)

    return p_value


def run_backtesting(file_path, VaR_final_table,alpha_var):

    data = pd.read_excel(file_path)

    data['Date'] = pd.to_datetime(data['Date'])
    data = data.set_index('Date').sort_index()
    #.asfreq('B').ffill()

    results = []

    for _, row in VaR_final_table.iterrows():

        asset = row['Asset']
        var_value = row['VaR_final']
        model_name = row['VaR_model']

        returns = np.log(data[asset] / data[asset].shift(1)).dropna()

        n = len(returns)

        x = (returns < var_value).sum()

        p_kupiec = kupiec_test(n, x, alpha_var)
        kupiec_valid = "Yes" if p_kupiec > 0.05 else "No"

        var_series = pd.Series(var_value, index=returns.index)

        p_christo = christoffersen_test(returns, var_series)
        christo_valid = "Yes" if p_christo > 0.05 else "No"

        results.append([
            asset,
            model_name,
            var_value,
            n,
            x,
            kupiec_valid,
            christo_valid
        ])

    backtest_table = pd.DataFrame(
        results,
        columns=[
            "Asset",
            "VaR_model",
            "VaR_final",
            "Days",
            "Violations",
            "VaR_Valid_Kupiec",
            "VaR_Valid_Christoffersen"
        ]
    )

    return backtest_table

VaR_table = run_var_garch_et_gjr_analysis("Data_set.xlsx")

VaR_final_table = compute_var_final_table(VaR_table)

backtest_table = run_backtesting("Data_set.xlsx", VaR_final_table, alpha_var)

print(backtest_table)